# 07 · Group 1 Tool Contracts & Group 3 FastAPI Bridge

Creates real ASGI endpoints using notebook-defined logic. The tiny `api_entrypoint.py` is the only required .py file, because Uvicorn imports an ASGI Python module. Development API deliberately binds to localhost and fails closed in production mode.

In [ ]:
%run 04_research_and_analytics.ipynb

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException, Query, Depends
from fastapi.responses import JSONResponse
from uuid import UUID
from typing import Optional
app=FastAPI(title='Group 2 Notebook-First RAG & Recommendation API',version='0.2.0')

def dev_access(project_id: str):
    if os.getenv('GROUP2_DEV_MODE','false').lower()!='true':
        raise HTTPException(503,'Authentication adapter not integrated: deny by default')
    try:return ensure_project_id(project_id)
    except Exception:raise HTTPException(422,'Invalid project UUID')

@app.get('/health')
def health():return {'status':'ok','mode':'dev-only','db':'sqlite'}

@app.post('/api/v1/projects/{project_id}/documents')
async def upload_document(project_id:str,domain:str,file:UploadFile=File(...),project_scope:str=Depends(dev_access)):
    try:
        if domain not in DOMAINS:raise ValueError('Unknown domain')
        data=await file.read(MAX_UPLOAD+1)
        return ingest_blob(project_scope,domain,file.filename or '',data)
    except (ValueError,UnicodeError,RuntimeError) as exc:raise HTTPException(400,str(exc)) from exc
    finally:await file.close()

@app.get('/api/v1/projects/{project_id}/documents')
def list_documents(project_id:str,domain:Optional[str]=None,project_scope:str=Depends(dev_access)):
    if domain and domain not in DOMAINS:raise HTTPException(422,'Unknown domain')
    q='SELECT id,filename,domain,status,source_url FROM documents WHERE project_id=?'
    params=[project_scope]
    if domain:q+=' AND domain=?';params.append(domain)
    with db_connect() as cx:return [dict(r) for r in cx.execute(q+' ORDER BY created_at DESC LIMIT 100',params)]

@app.post('/api/v1/projects/{project_id}/knowledge/search')
def knowledge_search(project_id:str,payload:dict,project_scope:str=Depends(dev_access)):
    try:
        validated=SearchInput.model_validate({**payload,'project_id':project_scope})
        return {'query':validated.question,
                'evidence':hybrid_search(project_scope,validated.question,validated.domain,validated.top_k),
                'retrieval_method':'hybrid-tfidf-bm25; offline baseline'}
    except (ValueError,KeyError) as exc:raise HTTPException(422,str(exc)) from exc

@app.post('/api/v1/projects/{project_id}/recommendations/rank')
def recommend(project_id:str,payload:dict,project_scope:str=Depends(dev_access)):
    try:return rank_candidates({**payload,'project_id':project_scope})
    except ValueError as exc:raise HTTPException(422,str(exc)) from exc

# Pure tool contract for Group 1. Reuse functions instead of importing agents.
def agent_knowledge_tool(project_id,domain,query,top_k=5):
    return {'evidence':hybrid_search(project_id,query,domain,top_k)}
def agent_recommendation_tool(payload):return rank_candidates(payload)

print('FastAPI routes:',[r.path for r in app.routes if hasattr(r,'methods')])

@app.post('/api/v1/projects/{project_id}/research/tasks')
def api_research(project_id:str,payload:dict,project_scope:str=Depends(dev_access)):
    try:
        return run_research_task(project_scope,payload['domain'],payload['question'],
            payload.get('source_mode','indexed'),payload.get('urls'),payload.get('top_k',5))
    except (ValueError,KeyError) as exc:raise HTTPException(422,str(exc)) from exc

@app.get('/api/v1/projects/{project_id}/research/tasks/{task_id}')
def api_get_research(project_id:str,task_id:str,project_scope:str=Depends(dev_access)):
    try:out=retrieve_research_task(project_scope,task_id)
    except ValueError as exc:raise HTTPException(422,str(exc)) from exc
    if out is None:raise HTTPException(404,'Research task not found in this project')
    return out

@app.post('/api/v1/projects/{project_id}/analytics/financial')
def api_financial(project_id:str,payload:dict,project_scope:str=Depends(dev_access)):
    try:return financial_metrics(payload['rows'],project_scope,payload.get('source_name','user-uploaded'),
                                 payload.get('currency','USD'))
    except (ValueError,KeyError,TypeError) as exc:raise HTTPException(422,str(exc)) from exc

def agent_research_tool(project_id,domain,question,source_mode='indexed',urls=None,top_k=5):
    return run_research_task(project_id,domain,question,source_mode,urls,top_k)

def agent_financial_tool(project_id,records,source_name):
    return financial_metrics(records,project_id,source_name)


## Market Basket Module (Apriori, CMPE 256 retail example)

Independent purchase-pattern recommender; deliberately separated from document-RAG technical/clinical recommendations. All routes apply existing development project scope checks. Group 3 must add production identity authorization.

In [ ]:
%run 14_market_basket_engine.ipynb

In [ ]:
from pydantic import ValidationError

@app.get('/api/v1/projects/{project_id}/market-basket/products')
def api_basket_products(project_id:str,project_scope:str=Depends(dev_access)):
    return basket_inventory(project_scope)

@app.post('/api/v1/projects/{project_id}/market-basket/recommend')
def api_basket_recommend(project_id:str,payload:BasketRecommendInput,project_scope:str=Depends(dev_access)):
    try:return market_basket_recommend(project_scope,**payload.model_dump())
    except (ValueError,ValidationError) as exc:raise HTTPException(422,str(exc)) from exc

@app.post('/api/v1/projects/{project_id}/market-basket/transactions/import')
def api_basket_import(project_id:str,payload:BasketImportInput,project_scope:str=Depends(dev_access)):
    try:
        return import_basket_rows(project_scope,[row.model_dump() for row in payload.rows],
                                  source_type='USER_DATASET',replace=payload.replace)
    except (ValueError,KeyError,ValidationError) as exc:raise HTTPException(422,str(exc)) from exc

def agent_market_basket_tool(project_id,cart,top_k=5):
    return market_basket_recommend(project_id,cart,top_k=top_k)

print('Market basket REST routes registered in Group 2 FastAPI app')